# Exercise 01: FGSM Attack — SOLUTION

**Goal:** Implement the Fast Gradient Sign Method (FGSM) and evaluate its effectiveness at different perturbation strengths.

**Estimated time:** ~20 minutes

**Reference:** Goodfellow et al. 2014 — *Explaining and Harnessing Adversarial Examples*

## Background

FGSM is the simplest adversarial attack. Given an image `x`, true label `y`, and a model with parameters `θ`, it computes:

```
x_adv = x + ε · sign( ∇_x L(θ, x, y) )
```

The key idea: move in the direction that **increases the loss** by a small amount `ε`. The `sign()` function normalizes the gradient so every pixel is perturbed by exactly `ε`.

**Steps:**
1. Enable gradient tracking on the input image
2. Forward pass through the model, compute cross-entropy loss
3. Backward pass to get `∇_x L`
4. Apply perturbation: `x_adv = x + ε * sign(∇_x L)`
5. Clip to valid pixel range

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

# Load pretrained ResNet20 trained on CIFAR-10, set to eval mode
model = torch.hub.load('chenyaofo/pytorch-cifar-models', 'cifar10_resnet20', pretrained=True)
model.eval()

# Load 100 CIFAR-10 test images (32x32, no resize needed)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616])
])
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=100, shuffle=False)
images, labels = next(iter(testloader))

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']

def predict(model, x):
    with torch.no_grad():
        logits = model(x)
    return logits.argmax(dim=1)

# Evaluate clean accuracy
clean_preds = predict(model, images)
print(f"Clean accuracy: {(clean_preds == labels).float().mean():.2%}")

In [ ]:
# SOLUTION
def fgsm_attack(model, images, labels, epsilon):
    """
    FGSM: x_adv = x + epsilon * sign(grad_x Loss(model(x), labels))
    """
    # 1. Create leaf tensor with gradient tracking
    x = images.clone().detach().requires_grad_(True)
    # 2. Forward pass + loss
    loss = nn.CrossEntropyLoss()(model(x), labels)
    # 3. Backward to get gradient w.r.t. input
    loss.backward()
    # 4. Apply signed gradient perturbation
    x_adv = x + epsilon * x.grad.sign()
    # 5. Clamp to the approximate valid range for ImageNet-normalized images
    x_adv = torch.clamp(x_adv, images.min().item(), images.max().item())
    return x_adv.detach()

In [ ]:
# SOLUTION — Evaluate at multiple epsilon values
epsilons = [0.01, 0.05, 0.1, 0.3, 0.5]
adv_accuracies = []

print(f"{'Epsilon':>10}  {'Adv Accuracy':>14}")
print("-" * 28)
for eps in epsilons:
    adv_images = fgsm_attack(model, images, labels, eps)
    adv_preds = predict(model, adv_images)
    acc = (adv_preds == labels).float().mean().item()
    adv_accuracies.append(acc)
    print(f"{eps:>10.3f}  {acc:>14.2%}")

In [ ]:
# SOLUTION — Plot accuracy vs epsilon
clean_acc = (clean_preds == labels).float().mean().item()

plt.figure(figsize=(8, 5))
plt.plot(epsilons, adv_accuracies, 'o-', color='red', linewidth=2, markersize=8, label='Adversarial accuracy')
plt.axhline(y=clean_acc, color='blue', linestyle='--', linewidth=2, label=f'Clean accuracy ({clean_acc:.2%})')
plt.xlabel('Epsilon (perturbation magnitude)')
plt.ylabel('Accuracy')
plt.title('FGSM: Accuracy vs. Perturbation Strength (ResNet20, CIFAR-10)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.ylim(0, 1)
plt.tight_layout()
plt.show()
print("\nExpected: accuracy drops sharply as epsilon increases.")

In [ ]:
# SOLUTION — Visualize adversarial examples
def denormalize(tensor, mean=[0.4914, 0.4822, 0.4465], std=[0.2470, 0.2435, 0.2616]):
    """Reverse CIFAR-10 normalization for display."""
    t = tensor.clone()
    for i in range(3):
        t[i] = t[i] * std[i] + mean[i]
    return torch.clamp(t, 0, 1)

eps_viz = 0.1
adv_images_viz = fgsm_attack(model, images, labels, eps_viz)
adv_preds_viz = predict(model, adv_images_viz)

fig, axes = plt.subplots(3, 5, figsize=(15, 9))
row_titles = ['Original', 'Perturbation (×10)', 'Adversarial']
for row in range(3):
    axes[row, 0].set_ylabel(row_titles[row], fontsize=12)

for i in range(5):
    orig = denormalize(images[i]).permute(1, 2, 0).numpy()
    adv  = denormalize(adv_images_viz[i]).permute(1, 2, 0).numpy()
    pert = adv_images_viz[i] - images[i]
    pert_display = ((pert - pert.min()) / (pert.max() - pert.min() + 1e-8)).permute(1, 2, 0).numpy()

    axes[0, i].imshow(orig)
    axes[0, i].set_title(f'True: {CIFAR10_CLASSES[labels[i].item()]}', fontsize=10)
    axes[0, i].axis('off')

    axes[1, i].imshow(pert_display)
    axes[1, i].set_title('Noise (scaled)', fontsize=10)
    axes[1, i].axis('off')

    axes[2, i].imshow(adv)
    pred_label = CIFAR10_CLASSES[adv_preds_viz[i].item()]
    true_label = CIFAR10_CLASSES[labels[i].item()]
    color = 'red' if pred_label != true_label else 'green'
    axes[2, i].set_title(f'Pred: {pred_label}', fontsize=10, color=color)
    axes[2, i].axis('off')

plt.suptitle(f'FGSM Attack (ε={eps_viz}) — Red title = attack succeeded', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()